# 00 · Triplane (TPV) lifting for every technique - overview

`lifting.triplane` gives **every** lifting technique in Halos-lift a triplane counterpart: instead of one bird's-eye
view map `(B, C, X, Y)` it produces three orthogonal planes

* `xy (B, C, X, Y)` - top view (this **is** the BEV map),
* `xz (B, C, X, Z)` - side view,
* `yz (B, C, Y, Z)` - front view,

and any 3D point is described by $f(x, y, z) = xy(x, y) + xz(x, z) + yz(y, z)$ (Huang et al., TPVFormer, CVPR 2023).

**The triplane module does not modify the existing lifters** (the only edits to existing code are the Alpamayo bug
fixes listed in `docs/triplane.md` → *Fixed issues*). The triplane module lives in its own package with its own registry
(`TRIPLANE_LIFTERS`); each triplane lifter *wraps* the existing BEV lifter (composition) and only adds what is needed to
keep the height axis. Every triplane lifter is still a `BaseLifter`, so `forward()` returns a BEV map and it plugs into
the existing `BEVSegmentationModel`, trainer and verifier unchanged.

| # | notebook | name | class | one-line idea |
|---|---|---|---|---|
| 01 | [`01_simple_bev_triplane.ipynb`](01_simple_bev_triplane.ipynb) | `simple_bev` | `SimpleBEVTriplaneLifter` | For every voxel: project its centre into every camera, bilinearly read the feature there, average over the cameras that see it - then collapse the volume along Z, Y and X. |
| 02 | [`02_depth_warp_triplane.ipynb`](02_depth_warp_triplane.ipynb) | `depth_warp` | `DepthWarpTriplaneLifter` | Predict a depth distribution for every pixel, build a depth-weighted frustum per camera, and let every voxel PULL its feature from the frustum at its (u, v, depth) - then collapse along Z, Y and X. |
| 03 | [`03_lift_splat_triplane.ipynb`](03_lift_splat_triplane.ipynb) | `lift_splat` | `LiftSplatTriplaneLifter` | Lift every pixel to D weighted 3D points (depth probability × context), splat (scatter-sum) the point cloud into voxels, and shoot by collapsing the volume along Z, Y and X. |
| 04 | [`04_tiim_triplane.ipynb`](04_tiim_triplane.ipynb) | `tiim` | `PolarRayTriplaneLifter` | A transformer turns every image column into a polar ray of depth cells; every voxel reads the ray at its (column, range) AND the image at its (column, row) for height - then collapse along Z, Y and X. |
| 05 | [`05_bevformer_triplane.ipynb`](05_bevformer_triplane.ipynb) | `bevformer` | `BEVFormerTriplaneLifter` | One BEVFormer spatial encoder per plane: learned queries on xy, xz and yz, each owning a line of 3D anchors along its plane normal, read the cameras with deformable spatial cross-attention. |
| 06 | [`06_tpvformer_triplane.ipynb`](06_tpvformer_triplane.ipynb) | `tpvformer` | `TPVFormerTriplaneLifter` | Natively a triplane lifter: XY + XZ + YZ learned query tokens alternate cross-view hybrid attention (planes talk to each other) and image cross-attention (anchors projected into the cameras). |
| 07 | [`07_alpamayo_triplane.ipynb`](07_alpamayo_triplane.ipynb) | `alpamayo` | `AlpamayoTriplaneAdapter` | TPVFormer layers whose plane cells follow a Mip-NeRF 360 non-linear parametrisation - dense near the ego, coarse far away - and whose planes are patchified into a fixed-length token sequence for a VLA policy. |
| 08 | [`08_mlp_view_triplane.ipynb`](08_mlp_view_triplane.ipynb) | `mlp_view` | `MLPViewTriplaneLifter` | One fully connected layer per (camera, plane) maps the image cells straight to plane cells; calibration is ignored - the control that proves every other triplane lifter really uses the cameras. |
| 09 | [`09_bev_adapter_triplane.ipynb`](09_bev_adapter_triplane.ipynb) | `bev_adapter` | `BEVLifterTriplaneAdapter` | Run any registered BEV lifter unchanged, then expand its BEV map into three planes with the learned BEVToTPV: xy from the map, xz / yz from pooled columns expanded along height. |

## Why three planes?

| representation | cells for `200 × 200 × 16` | keeps height? |
|---|---|---|
| voxels | 640,000 | yes |
| BEV | 40,000 | **no** - a pedestrian and the sign above them share one cell |
| **TPV** | 46,400 (13.8× fewer than voxels) | **yes**, factorised over three planes |

## Architecture of the triplane module

How the new registry relates to the existing one (dotted = composition: the triplane lifter holds the unchanged BEV
lifter as `self.core`):

```mermaid
flowchart TB
  subgraph EXIST["lifting.lifters · LIFTERS registry (existing, unchanged)"]
    direction LR
    L1[simple_bev]:::reused
    L2[depth_warp]:::reused
    L3[lift_splat]:::reused
    L4[tiim]:::reused
    L5[bevformer]:::reused
    L6[tpvformer]:::reused
    L7[alpamayo]:::reused
    L8[mlp_view]:::reused
  end
  subgraph NEW["lifting.triplane · TRIPLANE_LIFTERS registry (new)"]
    direction LR
    T1["SimpleBEVTriplaneLifter"]:::learned
    T2["DepthWarpTriplaneLifter"]:::learned
    T3["LiftSplatTriplaneLifter"]:::learned
    T4["PolarRayTriplaneLifter"]:::learned
    T5["BEVFormerTriplaneLifter"]:::learned
    T6["TPVFormerTriplaneLifter"]:::learned
    T7["AlpamayoTriplaneAdapter"]:::learned
    T8["MLPViewTriplaneLifter"]:::learned
    T9["BEVLifterTriplaneAdapter"]:::learned
  end
  L1 -. "core (composition)" .-> T1
  L2 -. core .-> T2
  L3 -. core .-> T3
  L4 -. core .-> T4
  L6 -. core .-> T6
  L7 -. core .-> T7
  EXIST -. "any entry" .-> T9
  BASE["BaseTriplaneLifter(BaseLifter)<br/>lift_planes() → TPVPlanes<br/>forward() → BEV from planes"]:::geo
  NEW --> BASE
  BASE --> OCC["build_triplane_model → TriplaneOccupancyModel<br/>3D occupancy + point queries"]:::out
  BASE --> BEVM["build_triplane_bev_model → BEVSegmentationModel<br/>BEV segmentation, existing trainer / verifier"]:::out

  classDef data fill:#fff7e6,stroke:#d48806,color:#222
  classDef geo fill:#eaf2fb,stroke:#1f77b4,color:#222
  classDef learned fill:#f1e6f8,stroke:#8e44ad,color:#222
  classDef reused fill:#eaf7ea,stroke:#2e8b57,color:#222
  classDef out fill:#fdecea,stroke:#c0392b,color:#222
```

The class hierarchy (SOLID: one contract - `lift_planes` - with a template-method `forward`; one class per file;
new behaviour added by new classes, not by editing old ones):

```mermaid
classDiagram
  direction LR
  class BaseLifter {
    <<existing ABC>>
    +grid: GridSpec
    +uses_geometry: bool
    +forward(features, cameras) Tensor BEV
  }
  class BaseTriplaneLifter {
    <<new ABC>>
    +bev_head: TriplaneBEVHead
    +lift_planes(features, cameras)* TPVPlanes
    +forward(features, cameras) Tensor BEV
    +resolve_kwargs(kwargs, context) dict
  }
  class VoxelToTriplane {
    +forward(voxels) TPVPlanes
  }
  class PlaneQueryEncoder {
    +forward(values, shapes, ref, visible) Tensor
  }
  class PlaneProjection {
    +forward(planes) TPVPlanes
  }
  class TriplaneBEVHead {
    +forward(planes) Tensor
  }
  class TriplaneOccupancyModel {
    +forward(images, cameras, points) dict
  }
  BaseLifter <|-- BaseTriplaneLifter
  BaseTriplaneLifter <|-- SimpleBEVTriplaneLifter
  BaseTriplaneLifter <|-- DepthWarpTriplaneLifter
  BaseTriplaneLifter <|-- LiftSplatTriplaneLifter
  BaseTriplaneLifter <|-- PolarRayTriplaneLifter
  BaseTriplaneLifter <|-- BEVFormerTriplaneLifter
  BaseTriplaneLifter <|-- TPVFormerTriplaneLifter
  BaseTriplaneLifter <|-- AlpamayoTriplaneAdapter
  BaseTriplaneLifter <|-- MLPViewTriplaneLifter
  BaseTriplaneLifter <|-- BEVLifterTriplaneAdapter
  BaseTriplaneLifter *-- TriplaneBEVHead
  SimpleBEVTriplaneLifter *-- VoxelToTriplane
  DepthWarpTriplaneLifter *-- VoxelToTriplane
  LiftSplatTriplaneLifter *-- VoxelToTriplane
  PolarRayTriplaneLifter *-- VoxelToTriplane
  BEVFormerTriplaneLifter *-- PlaneQueryEncoder
  BEVFormerTriplaneLifter *-- PlaneProjection
  TPVFormerTriplaneLifter *-- PlaneProjection
  AlpamayoTriplaneAdapter *-- PlaneProjection
  TriplaneOccupancyModel o-- BaseTriplaneLifter
```

## Every technique, animated

Each GIF plays the lifting steps of one technique on a toy scene (4 cameras, a car, a pedestrian, a pole); they are
rendered by `lifting.triplane.animation` (`python scripts/make_triplane_gifs.py`). Each technique notebook shows the
same animation, every step as a still image, and a detailed Mermaid diagram.

| **Simple-BEV** (`simple_bev`) |
|:--:|
| ![simple_bev](../../assets/triplane/simple_bev.gif) |

| **Depth-Warp** (`depth_warp`) |
|:--:|
| ![depth_warp](../../assets/triplane/depth_warp.gif) |

| **Lift-Splat-Shoot** (`lift_splat`) |
|:--:|
| ![lift_splat](../../assets/triplane/lift_splat.gif) |

| **TIIM (polar rays)** (`tiim`) |
|:--:|
| ![tiim](../../assets/triplane/tiim.gif) |

| **BEVFormer (V1 + V2 multi-scale)** (`bevformer`) |
|:--:|
| ![bevformer](../../assets/triplane/bevformer.gif) |

| **TPVFormer** (`tpvformer`) |
|:--:|
| ![tpvformer](../../assets/triplane/tpvformer.gif) |

| **Alpamayo (Mip-NeRF 360 triplane)** (`alpamayo`) |
|:--:|
| ![alpamayo](../../assets/triplane/alpamayo.gif) |

| **MLP view transform (negative control)** (`mlp_view`) |
|:--:|
| ![mlp_view](../../assets/triplane/mlp_view.gif) |

| **BEV-lifter adapter (any LIFTERS entry → triplane)** (`bev_adapter`) |
|:--:|
| ![bev_adapter](../../assets/triplane/bev_adapter.gif) |

In [1]:
import os, sys, time, warnings
warnings.filterwarnings("ignore")

# tpv_kit lives next to this notebook; also works when Jupyter starts at the repo root.
for _p in (".", os.path.join("notebooks", "triplane")):
    if os.path.isdir(os.path.join(_p, "tpv_kit")):
        sys.path.insert(0, os.path.abspath(_p)); break

import numpy as np
import torch, torch.nn.functional as F
import matplotlib.pyplot as plt

from lifting import BaseLifter
from lifting.encoders import SimpleConvEncoder
from lifting.lifters import LIFTERS
from lifting.triplane import TRIPLANE_LIFTERS, build_triplane_lifter, build_triplane_model, build_triplane_bev_model
from tpv_kit import (TriplaneScene, PlanePlotter, PipelineDiagram, StoryboardViewer,
                     TriplaneContractChecker, OccupancyExperiment, ExperimentResult)

torch.manual_seed(0)
torch.set_num_threads(min(4, torch.get_num_threads()))
plt.rcParams.update({"figure.dpi": 100, "image.interpolation": "nearest"})
plotter = PlanePlotter()
print("torch", torch.__version__, "| triplane lifters:", TRIPLANE_LIFTERS.names())

torch 2.14.1 | triplane lifters: ['alpamayo', 'bev_adapter', 'bevformer', 'depth_warp', 'lift_splat', 'mlp_view', 'simple_bev', 'tiim', 'tpvformer']


## 1. Every BEV lifter has a triplane counterpart

In [2]:
missing = set(LIFTERS.names()) - set(TRIPLANE_LIFTERS.names())
print("LIFTERS          :", LIFTERS.names())
print("TRIPLANE_LIFTERS :", TRIPLANE_LIFTERS.names())
print("BEV lifters without a triplane counterpart:", missing or "none")
print("extra: 'bev_adapter' wraps ANY LIFTERS entry, including user plug-ins")
assert not missing

LIFTERS          : ['alpamayo', 'bevformer', 'depth_warp', 'lift_splat', 'mlp_view', 'simple_bev', 'tiim', 'tpvformer']
TRIPLANE_LIFTERS : ['alpamayo', 'bev_adapter', 'bevformer', 'depth_warp', 'lift_splat', 'mlp_view', 'simple_bev', 'tiim', 'tpvformer']
BEV lifters without a triplane counterpart: none
extra: 'bev_adapter' wraps ANY LIFTERS entry, including user plug-ins


## 2. The contract, verified for all of them

Five executable checks per lifter (plane shapes, BEV read off the planes, `f = xy + xz + yz`, gradients reach the
images, the wrapped core is the unchanged registered class).

In [3]:
scene = TriplaneScene()
print(scene.describe())
batch = scene.sample
C = 32
encoder = SimpleConvEncoder(out_channels=C, num_levels=2)
features = scene.encode(encoder, batch)
SMALL = {"bevformer": {"num_layers": 1}, "tpvformer": {"num_layers": 1}, "alpamayo": {"num_layers": 1}}
ok = {}
for name in TRIPLANE_LIFTERS.names():
    print(f"\n=== {name}")
    lifter = build_triplane_lifter(name, scene.grid, C, scene.image_size, scene.num_cameras, encoder, **SMALL.get(name, {}))
    ok[name] = TriplaneContractChecker(lifter, features, batch.cameras).run()
print("\nall contracts hold:", all(ok.values()))

grid      : x (-16.0, 16.0) y (-16.0, 16.0) z (-1.0, 3.0) m, resolution (16, 16, 4)
voxel     : (2.0, 2.0, 1.0) m
cameras   : 4 x (48, 96) px
scenes    : 128 train / 32 test (batch 4)
TPV cells : xy 256 + xz 64 + yz 64 = 384  (voxels: 1024)

=== alpamayo
PASS  plane shapes            xy(4, 32, 16, 16) xz(4, 32, 16, 4) yz(4, 32, 16, 4)
PASS  BEV = bev_head(planes)  BEV (4, 32, 16, 16), max |diff| = 0.0e+00
PASS  f = xy + xz + yz        voxel (8,8,2) max |diff| = 0.0e+00
PASS  reuses BEV lifter       core = AlpamayoTriplaneLifter = LIFTERS['alpamayo']
PASS  gradients reach images  1/2 pyramid levels receive gradient

=== bev_adapter
PASS  plane shapes            xy(4, 32, 16, 16) xz(4, 32, 16, 4) yz(4, 32, 16, 4)
PASS  BEV = bev_head(planes)  BEV (4, 32, 16, 16), max |diff| = 0.0e+00
PASS  f = xy + xz + yz        voxel (8,8,2) max |diff| = 0.0e+00
PASS  reuses BEV lifter       core = BilinearSamplingLifter = LIFTERS['simple_bev']
PASS  gradients reach images  1/2 pyramid levels receive g

PASS  plane shapes            xy(4, 32, 16, 16) xz(4, 32, 16, 4) yz(4, 32, 16, 4)
PASS  BEV = bev_head(planes)  BEV (4, 32, 16, 16), max |diff| = 0.0e+00
PASS  f = xy + xz + yz        voxel (8,8,2) max |diff| = 0.0e+00
PASS  reuses BEV lifter       core = LiftSplatLifter = LIFTERS['lift_splat']
PASS  gradients reach images  1/2 pyramid levels receive gradient

=== mlp_view
PASS  plane shapes            xy(4, 32, 16, 16) xz(4, 32, 16, 4) yz(4, 32, 16, 4)
PASS  BEV = bev_head(planes)  BEV (4, 32, 16, 16), max |diff| = 0.0e+00
PASS  f = xy + xz + yz        voxel (8,8,2) max |diff| = 0.0e+00
PASS  reuses BEV lifter       native triplane lifter built from existing layers (MLPViewTriplaneLifter)
PASS  gradients reach images  1/2 pyramid levels receive gradient

=== simple_bev
PASS  plane shapes            xy(4, 32, 16, 16) xz(4, 32, 16, 4) yz(4, 32, 16, 4)
PASS  BEV = bev_head(planes)  BEV (4, 32, 16, 16), max |diff| = 0.0e+00
PASS  f = xy + xz + yz        voxel (8,8,2) max |diff| = 0.0e+00


## 3. Train every triplane lifter on 3D occupancy

Same scenes, same budget (150 AdamW steps, 128 training scenes, CPU), per-plane auxiliary loss on. `corrupted` = the
same trained model evaluated with every camera yawed by 90°: geometric lifters must collapse, the geometry-free
`mlp_view` control cannot localise objects at all on these randomly rotated rigs.

In [4]:
results, predictions = [], {}
for name in TRIPLANE_LIFTERS.names():
    exp = OccupancyExperiment(scene, name, **SMALL.get(name, {}))
    results.append(exp.run(steps=150))
    predictions[name] = exp.predict()["voxel_logits"].argmax(1)
print(ExperimentResult.header())
for r in sorted(results, key=lambda r: -r.miou_after):
    print(r.row())

lifter          params    time   before    after  corrupted   verdict
lift_splat        521k    5.5s    0.004    0.388      0.028   uses geometry
tiim              865k    6.8s    0.004    0.386      0.031   uses geometry
depth_warp        521k    5.1s    0.004    0.386      0.029   uses geometry
bev_adapter       240k    7.0s    0.013    0.381      0.031   uses geometry
simple_bev        510k    4.3s    0.000    0.375      0.030   uses geometry
alpamayo          236k    7.8s    0.018    0.344      0.030   uses geometry
tpvformer         236k    5.5s    0.017    0.324      0.030   uses geometry
bevformer         249k    6.5s    0.008    0.295      0.030   uses geometry
mlp_view          318k    3.4s    0.013    0.038      0.038   cannot localise objects


In [5]:
names = [r.name for r in results]
x = np.arange(len(results))
fig, ax = plt.subplots(figsize=(11, 3.6), facecolor="white")
for k, (attr, label, color) in enumerate((("miou_before", "untrained", "#bbbbbb"),
                                         ("miou_after", "trained", "#8e44ad"),
                                         ("miou_corrupted", "trained, cameras yawed 90°", "#e8710a"))):
    ax.bar(x + (k - 1) * 0.27, [getattr(r, attr) for r in results], 0.27, label=label, color=color)
ax.set_xticks(x, names, rotation=20); ax.set_ylabel("foreground mIoU (3D occupancy)")
ax.set_title("every triplane lifter: learns with correct calibration, collapses with wrong calibration", fontsize=11)
ax.set_ylim(0, 1.3 * max(r.miou_after for r in results))
ax.legend(fontsize=8.5, ncol=3, loc="upper center"); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()
OccupancyExperiment.plot_losses(results, title="occupancy training loss, all triplane lifters")

In [6]:
small = PlanePlotter(row_height=1.9)
rows = [("ground truth", scene.label_planes(batch.occupancy))]
rows += [(r.name, scene.label_planes(predictions[r.name])) for r in sorted(results, key=lambda r: -r.miou_after)]
small.labels(rows, title="the same held-out scene, predicted by every triplane lifter")

## 4. Which one should I use?

| need | pick | why |
|---|---|---|
| strongest geometry prior, no depth labels, cheap | `simple_bev` | parameter-free unprojection; planes grounded on all axes |
| depth labels available | `lift_splat` / `depth_warp` + `DepthSupervisedTask` | a supervised depth distribution concentrates features at the right range |
| long range, column-wise reasoning | `tiim` | transformer per image column, range + elevation per voxel |
| learned queries, flexible capacity, multi-scale | `bevformer` (V1 / V2 with `num_levels=2`) | deformable attention, independent planes |
| best query-based triplane, planes share information | `tpvformer` | cross-view hybrid attention |
| VLA tokenizer | `alpamayo` | `patchify_planes` → fixed-length tokens |
| sanity check | `mlp_view` | geometry-free floor every lifter must beat |
| your own BEV lifter | `bev_adapter` | wraps any `LIFTERS` entry |

## 5. Using it

```python
from lifting.triplane import TRIPLANE_LIFTERS, build_triplane_model, build_triplane_bev_model

lifter = TRIPLANE_LIFTERS.build("lift_splat", grid=grid, in_channels=64, out_channels=64)
planes = lifter.lift_planes(features, cameras)          # TPVPlanes(xy, xz, yz)
bev    = lifter(features, cameras)                      # (B, 64, X, Y): still a BEV lifter

model = build_triplane_model("tiim", grid, num_classes=3, aux_heads=True)       # 3D occupancy + points
bev_model = build_triplane_bev_model("bevformer", grid, num_classes=3, num_levels=2)  # BEV seg, BEVFormer V2 triplane
```

Next: open any technique notebook `01` - `09` for the step-by-step version.